# Si/Ge (001) Strained Superlattice

## 0. Introduction

This tutorial builds the Si/Ge (001) superlattice and its two bulk reference cells from the publication:

> **Chris G. Van de Walle and Richard M. Martin**, "Theoretical calculations of heterojunction discontinuities in
> the Si/Ge system", Physical Review B 34, 5621 (1986).
> [DOI: 10.1103/PhysRevB.34.5621](https://doi.org/10.1103/PhysRevB.34.5621)

Germanium is grown pseudomorphically on silicon (001): both materials share silicon's in-plane lattice constant,
5.43 Å; silicon stays cubic and germanium is stretched out of plane to 5.82 Å (Table I). The superlattice has four
Si and four Ge (001) planes, one atom per plane, two identical interfaces and no vacuum (Fig. 1). The spacing at
each interface is the mean of the two bulk plane spacings (Sec. II), and the atoms stay at these ideal positions.

The valence band offset is computed from this superlattice and from separate bulk cells of cubic Si and of Ge
strained as in the superlattice (Sec. III.A), so all three are saved:
[the simulation notebook](interface_3d_3d_silicon_germanium_SIMULATION.ipynb) loads them by name.

## 1. Prepare the Environment
### 1.1. Set up the parameters

In [ ]:
SUBSTRATE_NAME = "mp-149"  # Si, Standata
FILM_NAME = "mp-32"  # Ge, Standata
MILLER_INDICES = (0, 0, 1)
NUMBER_OF_LAYERS = 1  # conventional (001) layers, four atomic planes each

IN_PLANE_LATTICE_CONSTANT = 5.43  # Angstrom, silicon's, shared by both materials
SUBSTRATE_OUT_OF_PLANE_LATTICE_CONSTANT = 5.43  # Angstrom, cubic Si
FILM_OUT_OF_PLANE_LATTICE_CONSTANT = 5.82  # Angstrom, Ge on Si(001), Table I

SUPERLATTICE_NAME = "Si/Ge (001) superlattice 4+4"
SUBSTRATE_BULK_NAME = "Si (001) bulk a=5.43"
FILM_BULK_NAME = "Ge (001) bulk a=5.43 c=5.82"

### 1.2. Install Packages
The step executes only in Pyodide environment. For other environments, the packages should be installed via `pip install` (see [README](../../README.ipynb)).

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|specific_examples")

### 1.3. Load the bulk materials and strain them

Each bulk is strained in its conventional cubic cell, before any slab is cut: the interface builder re-creates
its slabs from the bulk, so a strain applied to a slab would be lost. These strained cells, with the c axis along
[001], are also the bulk references of the simulation.

In [ ]:
from mat3ra.esse.models.core.abstract.matrix_3x3 import Matrix3x3Schema
from mat3ra.made.material import Material
from mat3ra.made.tools.analyze.lattice.helpers import get_material_with_conventional_lattice
from mat3ra.made.tools.operations.core.unary import strain
from mat3ra.standata.materials import Materials


def strain_conventional_cell(material, in_plane_lattice_constant, out_of_plane_lattice_constant):
    conventional_cell = get_material_with_conventional_lattice(material)
    in_plane_ratio = in_plane_lattice_constant / conventional_cell.lattice.a
    out_of_plane_ratio = out_of_plane_lattice_constant / conventional_cell.lattice.c
    strain_matrix = [[in_plane_ratio, 0, 0], [0, in_plane_ratio, 0], [0, 0, out_of_plane_ratio]]
    return strain(conventional_cell, Matrix3x3Schema(root=strain_matrix))


substrate_bulk = strain_conventional_cell(Material.create(Materials.get_by_name_first_match(SUBSTRATE_NAME)),
                                          IN_PLANE_LATTICE_CONSTANT, SUBSTRATE_OUT_OF_PLANE_LATTICE_CONSTANT)
film_bulk = strain_conventional_cell(Material.create(Materials.get_by_name_first_match(FILM_NAME)),
                                     IN_PLANE_LATTICE_CONSTANT, FILM_OUT_OF_PLANE_LATTICE_CONSTANT)
substrate_bulk.name = SUBSTRATE_BULK_NAME
film_bulk.name = FILM_BULK_NAME

## 2. Create the Superlattice
### 2.1. Create the (001) slabs, without vacuum

In [ ]:
from mat3ra.made.tools.helpers import create_slab

substrate_slab, film_slab = [
    create_slab(crystal=bulk, miller_indices=MILLER_INDICES, number_of_layers=NUMBER_OF_LAYERS,
                use_conventional_cell=False, vacuum=0.0)
    for bulk in (substrate_bulk, film_bulk)
]

### 2.2. Stack the slabs, set the period and reduce to the primitive cell

The film is placed on the substrate at the mean of the two plane spacings, d = (a_Si⊥ + a_Ge⊥) / 8. The second,
periodic interface is set by the cell height: c = a_Si⊥ + a_Ge⊥ per layer gives it the same spacing. The cell is
reduced to the primitive cell only after c is set; reducing first moves the interface spacing into the Ge block.

In [ ]:
from mat3ra.made.tools.analyze.lattice.helpers import get_material_with_primitive_lattice
from mat3ra.made.tools.helpers import create_interface_simple_between_slabs

interface_spacing = (SUBSTRATE_OUT_OF_PLANE_LATTICE_CONSTANT + FILM_OUT_OF_PLANE_LATTICE_CONSTANT) / 8
interface = create_interface_simple_between_slabs(substrate_slab=substrate_slab, film_slab=film_slab,
                                                  gap=interface_spacing, vacuum=0.0)
lattice_vectors = interface.lattice.vector_arrays
period = NUMBER_OF_LAYERS * (SUBSTRATE_OUT_OF_PLANE_LATTICE_CONSTANT + FILM_OUT_OF_PLANE_LATTICE_CONSTANT)
interface.set_lattice_vectors(lattice_vectors[0], lattice_vectors[1], [0.0, 0.0, period])
superlattice = get_material_with_primitive_lattice(interface)
superlattice.name = SUPERLATTICE_NAME

### 2.3. Print the plane spacings

Si planes are a_Si⊥ / 4 = 1.3575 Å apart, Ge planes a_Ge⊥ / 4 = 1.455 Å, and both interfaces 1.406 Å.

In [ ]:
import numpy as np
from collections import Counter


def plane_spacings(material):
    cartesian_material = material.clone()
    cartesian_material.to_cartesian()
    plane_heights = np.unique(np.round(np.array(cartesian_material.basis.coordinates.values)[:, 2], 4))
    return np.diff(np.append(plane_heights, plane_heights[0] + cartesian_material.lattice.c))


for material in (superlattice, substrate_bulk, film_bulk):
    lattice = material.lattice
    print(f"{material.name}: {dict(Counter(material.basis.elements.values))}, {material.basis.number_of_atoms} atoms, "
          f"cell {lattice.a:.4f} x {lattice.b:.4f} x {lattice.c:.4f} Å")
    print(f"  plane spacings along c: {np.round(plane_spacings(material), 4).tolist()} Å")

## 3. Visualize the Result(s)

In [ ]:
from mat3ra.notebooks_utils.ipython.entity.material.visualize import visualize_materials

materials = [superlattice, substrate_bulk, film_bulk]
visualize_materials([{"material": material, "title": material.name, "rotation": "-90x"} for material in materials],
                    repetitions=[2, 2, 1])

## 4. Pass data to the outside runtime

In [ ]:
from mat3ra.notebooks_utils.material import set_materials

set_materials(materials)

## References

[1] C. G. Van de Walle and R. M. Martin, "Theoretical calculations of heterojunction discontinuities in the Si/Ge
system", Phys. Rev. B 34, 5621 (1986). [DOI: 10.1103/PhysRevB.34.5621](https://doi.org/10.1103/PhysRevB.34.5621)